## Цель 

Собрать рабочий baseline pipeline: 
1. чтение изображений из папки;
2. детекция таблички; 
3. OCR номера; 
4. нормализация результата; 
5. расчёт confidence; 
6. формирование нового имени файла; 
7. отчёт по обработке.

## Импорты

In [1]:
from pathlib import Path 
import random 
import re 
import shutil 
import cv2 
import matplotlib.pyplot as plt 
import numpy as np 
import pandas as pd 

from ultralytics import YOLO 
import easyocr

## Пути

In [2]:
PROJECT_ROOT = Path("..").resolve()

DATASET_DIR = PROJECT_ROOT / "dataset"
IMAGES_DIR = DATASET_DIR / "images"
MODEL_PATH = PROJECT_ROOT / "runs" / "detect" / "train4" / "weights" / "best.pt"
REPORTS_DIR = PROJECT_ROOT / "metadata"

REPORTS_DIR.mkdir(exist_ok=True)

print("MODEL_PATH exists:", MODEL_PATH.exists())
print("IMAGES_DIR exists:", IMAGES_DIR.exists())

MODEL_PATH exists: True
IMAGES_DIR exists: True


## Загрузка модели и OCR

In [3]:
model = YOLO(str(MODEL_PATH))
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


## Список изображений

In [4]:
image_paths = []

for split in ["train", "val"]:
    split_dir = IMAGES_DIR / split
    image_paths.extend([
        p for p in split_dir.glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])

print("Total images:", len(image_paths))

Total images: 123


## Базовые функции

In [10]:
def crop_bbox(image_bgr, bbox):
    h, w = image_bgr.shape[:2]
    x1, y1, x2, y2 = map(int, bbox)

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(w, x2)
    y2 = min(h, y2)

    return image_bgr[y1:y2, x1:x2]


def normalize_plate_orientation(image_bgr):
    h, w = image_bgr.shape[:2]
    if h > w:
        image_bgr = cv2.rotate(image_bgr, cv2.ROTATE_90_CLOCKWISE)
    return image_bgr


def extract_number_zone_wide(plate):
    h, w = plate.shape[:2]
    return plate[
        int(h * 0.35):int(h * 0.85),
        int(w * 0.05):int(w * 0.95)
    ]


def preprocess_blur_only(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blur, 130, 255, cv2.THRESH_BINARY)
    return th


def extract_text_from_easyocr(ocr_output):
    texts = []
    confs = []

    for item in ocr_output:
        bbox, text, conf = item
        texts.append(text)
        confs.append(conf)

    merged_text = " ".join(texts)
    mean_conf = float(np.mean(confs)) if confs else 0.0

    return merged_text, mean_conf


def process_ocr_text(raw_text: str) -> str:
    text = raw_text.lower()
    text = re.sub(r"[^0-9abcde]", "", text)

    match = re.fullmatch(r"\d{1,4}[abcde]?", text)
    return match.group(0) if match else ""


def normalize_plate_number(text: str) -> str:
    match = re.fullmatch(r"(\d{1,4})([abcde]?)", text)
    if not match:
        return ""

    digits, suffix = match.groups()
    digits = digits.zfill(4)
    return digits + suffix


def extract_expected_number_from_filename(filename: str) -> str:
    stem = Path(filename).stem.lower()
    match = re.search(r"(\d{1,4}[abcde]?)", stem)
    if not match:
        return ""
    return normalize_plate_number(match.group(1))

## Итоговый pipeline для одного изображения

In [11]:
def run_pipeline(image_path, model, reader):
    image_bgr = cv2.imread(str(image_path))

    det_res = model(str(image_path), verbose=False)[0]
    if len(det_res.boxes) == 0:
        return {
            "filename": image_path.name,
            "detected": False,
            "det_conf": 0.0,
            "ocr_raw": "",
            "ocr_clean": "",
            "ocr_norm": "",
            "ocr_conf": 0.0,
            "final_conf": 0.0,
            "status": "no_detection",
        }

    boxes_xyxy = det_res.boxes.xyxy.cpu().numpy()
    confidences = det_res.boxes.conf.cpu().numpy()
    best_idx = np.argmax(confidences)

    bbox = boxes_xyxy[best_idx]
    det_conf = float(confidences[best_idx])

    plate = crop_bbox(image_bgr, bbox)
    plate = normalize_plate_orientation(plate)

    zone = extract_number_zone_wide(plate)
    gray = cv2.cvtColor(zone, cv2.COLOR_BGR2GRAY)
    prep = preprocess_blur_only(gray)

    ocr_result = reader.readtext(prep)
    ocr_raw, ocr_conf = extract_text_from_easyocr(ocr_result)

    ocr_clean = process_ocr_text(ocr_raw)
    ocr_norm = normalize_plate_number(ocr_clean)

    final_conf = det_conf * ocr_conf

    status = "ok" if ocr_norm else "ocr_failed"

    return {
        "filename": image_path.name,
        "detected": True,
        "det_conf": det_conf,
        "ocr_raw": ocr_raw,
        "ocr_clean": ocr_clean,
        "ocr_norm": ocr_norm,
        "ocr_conf": ocr_conf,
        "final_conf": final_conf,
        "status": status,
    }

## Прогон

In [12]:
random.seed(42)
sample_paths = random.sample(image_paths, min(20, len(image_paths)))

rows = []

for path in sample_paths:
    res = run_pipeline(path, model, reader)
    expected = extract_expected_number_from_filename(path.name)

    res["expected"] = expected
    res["match"] = res["ocr_norm"] == expected if res["ocr_norm"] and expected else False

    rows.append(res)

df_sample = pd.DataFrame(rows)
df_sample

/home/alexander/venvs/plate-recognition/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


,filename,detected,det_conf,ocr_raw,ocr_clean,ocr_norm,ocr_conf,final_conf,status,expected,match
0,MDZ090 _9.jpg,True,0.921437,0 |ol,0,0000,0.081889,0.075456,ok,0090,False
1,TUL864_1.jpg,True,0.879056,81614,,,0.438428,0.385403,ocr_failed,0864,False
2,TUL803_3.jpg,True,0.948048,3 3 { 8 0 Cn,3380c,3380c,0.476616,0.451855,ok,0803,False
3,MDZ097_3.jpg,True,0.889849,,,,0.000000,0.000000,ocr_failed,0097,False
4,TUL082_6.jpg,True,0.942360,1 8 2,182,0182,0.920307,0.867260,ok,0082,False
5,TUL051_2.jpg,True,0.905069,5,5,0005,0.500000,0.452535,ok,0051,False
6,TUL023_6.jpg,True,0.984962,2 3,23,0023,0.867434,0.854389,ok,0023,True
7,TUL866.jpg,True,0.964316,8 61.6 / C,8616c,8616c,0.355502,0.342816,ok,0866,False
8,TUL859.jpg,True,0.860519,8 5 9,859,0859,0.572418,0.492577,ok,0859,True
9,MDZ094 _4.jpg,True,0.928574,9'4 |-,94,0094,0.526454,0.488851,ok,0094,True


In [13]:
df_sample[[
    "filename", "expected", "ocr_norm", "match",
    "det_conf", "ocr_conf", "final_conf", "status"
]]

,filename,expected,ocr_norm,match,det_conf,ocr_conf,final_conf,status
0,MDZ090 _9.jpg,0090,0000,False,0.921437,0.081889,0.075456,ok
1,TUL864_1.jpg,0864,,False,0.879056,0.438428,0.385403,ocr_failed
2,TUL803_3.jpg,0803,3380c,False,0.948048,0.476616,0.451855,ok
3,MDZ097_3.jpg,0097,,False,0.889849,0.000000,0.000000,ocr_failed
4,TUL082_6.jpg,0082,0182,False,0.942360,0.920307,0.867260,ok
5,TUL051_2.jpg,0051,0005,False,0.905069,0.500000,0.452535,ok
6,TUL023_6.jpg,0023,0023,True,0.984962,0.867434,0.854389,ok
7,TUL866.jpg,0866,8616c,False,0.964316,0.355502,0.342816,ok
8,TUL859.jpg,0859,0859,True,0.860519,0.572418,0.492577,ok
9,MDZ094 _4.jpg,0094,0094,True,0.928574,0.526454,0.488851,ok


In [14]:
df_sample["match"].mean()

np.float64(0.3)

## Логика безопасного автопереименования

In [15]:
def is_high_confidence(row, det_thr=0.90, ocr_thr=0.80):
    return (
        row["detected"] and
        row["det_conf"] >= det_thr and
        row["ocr_conf"] >= ocr_thr and
        row["ocr_norm"] != ""
    )

df_sample["high_confidence"] = df_sample.apply(is_high_confidence, axis=1)
df_sample

,filename,detected,det_conf,ocr_raw,ocr_clean,ocr_norm,ocr_conf,final_conf,status,expected,match,high_confidence
0,MDZ090 _9.jpg,True,0.921437,0 |ol,0,0000,0.081889,0.075456,ok,0090,False,False
1,TUL864_1.jpg,True,0.879056,81614,,,0.438428,0.385403,ocr_failed,0864,False,False
2,TUL803_3.jpg,True,0.948048,3 3 { 8 0 Cn,3380c,3380c,0.476616,0.451855,ok,0803,False,False
3,MDZ097_3.jpg,True,0.889849,,,,0.000000,0.000000,ocr_failed,0097,False,False
4,TUL082_6.jpg,True,0.942360,1 8 2,182,0182,0.920307,0.867260,ok,0082,False,True
5,TUL051_2.jpg,True,0.905069,5,5,0005,0.500000,0.452535,ok,0051,False,False
6,TUL023_6.jpg,True,0.984962,2 3,23,0023,0.867434,0.854389,ok,0023,True,True
7,TUL866.jpg,True,0.964316,8 61.6 / C,8616c,8616c,0.355502,0.342816,ok,0866,False,False
8,TUL859.jpg,True,0.860519,8 5 9,859,0859,0.572418,0.492577,ok,0859,True,False
9,MDZ094 _4.jpg,True,0.928574,9'4 |-,94,0094,0.526454,0.488851,ok,0094,True,False


In [16]:
def build_new_name(row):
    old_suffix = Path(row["filename"]).suffix.lower()

    if not row["ocr_norm"]:
        return row["filename"]

    new_name = row["ocr_norm"]

    if not row["high_confidence"]:
        new_name += "-!"

    return new_name + old_suffix

df_sample["new_name"] = df_sample.apply(build_new_name, axis=1)
df_sample

,filename,detected,det_conf,ocr_raw,ocr_clean,ocr_norm,ocr_conf,final_conf,status,expected,match,high_confidence,new_name
0,MDZ090 _9.jpg,True,0.921437,0 |ol,0,0000,0.081889,0.075456,ok,0090,False,False,0000-!.jpg
1,TUL864_1.jpg,True,0.879056,81614,,,0.438428,0.385403,ocr_failed,0864,False,False,TUL864_1.jpg
2,TUL803_3.jpg,True,0.948048,3 3 { 8 0 Cn,3380c,3380c,0.476616,0.451855,ok,0803,False,False,3380c-!.jpg
3,MDZ097_3.jpg,True,0.889849,,,,0.000000,0.000000,ocr_failed,0097,False,False,MDZ097_3.jpg
4,TUL082_6.jpg,True,0.942360,1 8 2,182,0182,0.920307,0.867260,ok,0082,False,True,0182.jpg
5,TUL051_2.jpg,True,0.905069,5,5,0005,0.500000,0.452535,ok,0051,False,False,0005-!.jpg
6,TUL023_6.jpg,True,0.984962,2 3,23,0023,0.867434,0.854389,ok,0023,True,True,0023.jpg
7,TUL866.jpg,True,0.964316,8 61.6 / C,8616c,8616c,0.355502,0.342816,ok,0866,False,False,8616c-!.jpg
8,TUL859.jpg,True,0.860519,8 5 9,859,0859,0.572418,0.492577,ok,0859,True,False,0859-!.jpg
9,MDZ094 _4.jpg,True,0.928574,9'4 |-,94,0094,0.526454,0.488851,ok,0094,True,False,0094-!.jpg


In [17]:
def add_duplicate_suffixes(df, base_name_col="new_name"):
    counters = {}
    final_names = []

    for _, row in df.iterrows():
        name = row[base_name_col]

        if name == row["filename"]:
            final_names.append(name)
            continue

        stem = Path(name).stem
        suffix = Path(name).suffix

        counters.setdefault(name, 0)
        counters[name] += 1

        if counters[name] == 1:
            final_names.append(name)
        else:
            final_names.append(f"{stem}_{counters[name]-1}{suffix}")

    return final_names

df_sample["final_name"] = add_duplicate_suffixes(df_sample)
df_sample

,filename,detected,det_conf,ocr_raw,ocr_clean,ocr_norm,ocr_conf,final_conf,status,expected,match,high_confidence,new_name,final_name
0,MDZ090 _9.jpg,True,0.921437,0 |ol,0,0000,0.081889,0.075456,ok,0090,False,False,0000-!.jpg,0000-!.jpg
1,TUL864_1.jpg,True,0.879056,81614,,,0.438428,0.385403,ocr_failed,0864,False,False,TUL864_1.jpg,TUL864_1.jpg
2,TUL803_3.jpg,True,0.948048,3 3 { 8 0 Cn,3380c,3380c,0.476616,0.451855,ok,0803,False,False,3380c-!.jpg,3380c-!.jpg
3,MDZ097_3.jpg,True,0.889849,,,,0.000000,0.000000,ocr_failed,0097,False,False,MDZ097_3.jpg,MDZ097_3.jpg
4,TUL082_6.jpg,True,0.942360,1 8 2,182,0182,0.920307,0.867260,ok,0082,False,True,0182.jpg,0182.jpg
5,TUL051_2.jpg,True,0.905069,5,5,0005,0.500000,0.452535,ok,0051,False,False,0005-!.jpg,0005-!.jpg
6,TUL023_6.jpg,True,0.984962,2 3,23,0023,0.867434,0.854389,ok,0023,True,True,0023.jpg,0023.jpg
7,TUL866.jpg,True,0.964316,8 61.6 / C,8616c,8616c,0.355502,0.342816,ok,0866,False,False,8616c-!.jpg,8616c-!.jpg
8,TUL859.jpg,True,0.860519,8 5 9,859,0859,0.572418,0.492577,ok,0859,True,False,0859-!.jpg,0859-!.jpg
9,MDZ094 _4.jpg,True,0.928574,9'4 |-,94,0094,0.526454,0.488851,ok,0094,True,False,0094-!.jpg,0094-!.jpg


In [18]:
df_sample = df_sample.sort_values(by="final_conf", ascending=True).reset_index(drop=True)
df_sample

,filename,detected,det_conf,ocr_raw,ocr_clean,ocr_norm,ocr_conf,final_conf,status,expected,match,high_confidence,new_name,final_name
0,MDZ097_3.jpg,True,0.889849,,,,0.000000,0.000000,ocr_failed,0097,False,False,MDZ097_3.jpg,MDZ097_3.jpg
1,QBA1096a_2.jpg,True,0.835561,,,,0.000000,0.000000,ocr_failed,1096a,False,False,QBA1096a_2.jpg,QBA1096a_2.jpg
2,MDZ090 _9.jpg,True,0.921437,0 |ol,0,0000,0.081889,0.075456,ok,0090,False,False,0000-!.jpg,0000-!.jpg
3,QBA1156_5.jpg,True,0.914066,9 G L [ 4 7,947,0947,0.297966,0.272361,ok,1156,False,False,0947-!.jpg,0947-!.jpg
4,TUL866.jpg,True,0.964316,8 61.6 / C,8616c,8616c,0.355502,0.342816,ok,0866,False,False,8616c-!.jpg,8616c-!.jpg
5,TUL864_1.jpg,True,0.879056,81614,,,0.438428,0.385403,ocr_failed,0864,False,False,TUL864_1.jpg,TUL864_1.jpg
6,TUL803_3.jpg,True,0.948048,3 3 { 8 0 Cn,3380c,3380c,0.476616,0.451855,ok,0803,False,False,3380c-!.jpg,3380c-!.jpg
7,TUL051_2.jpg,True,0.905069,5,5,0005,0.500000,0.452535,ok,0051,False,False,0005-!.jpg,0005-!.jpg
8,TUL855_5.jpg,True,0.585546,3 { 8,38,0038,0.790315,0.462766,ok,0855,False,False,0038-!.jpg,0038-!.jpg
9,MDZ087 _3.jpg,True,0.918648,8 ' 7,87,0087,0.522873,0.480336,ok,0087,True,False,0087-!.jpg,0087-!.jpg


In [19]:
report_path = REPORTS_DIR / "end_to_end_sample_report.csv"
df_sample.to_csv(report_path, index=False, encoding="utf-8-sig")

print("Saved:", report_path)

Saved: /media/alexander/6668CC7E68CC4E8D/Users/Alexander/projects/plate-recognition/metadata/end_to_end_sample_report.csv
